1 绑定1个自定义工具

In [14]:
from typing import Literal

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool


load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)


#2 声明1个工具
class weatherInput(BaseModel):
    """
    查询天气工具的形参的类的相关信息
    """
    city: str=Field(description="城市名称，例如'北京市','重庆市'",default="重庆市")
    unit:Literal["celsius","kelvin"]=Field(description="描述气温的单位")
    date:str=Field(description="查询日期")

@tool(description="查询具体城市的天气",args_schema=weatherInput)
def get_weather(city:str,unit:Literal["celsius","kelvin"],date:str):
    return f"{city} {date}的天气晴朗，气温10-16{unit}，有微风，适宜出行"
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    tools=[get_weather]
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                SystemMessage("你是1个私人ai助手"),
                HumanMessage("请你查询一下重庆2026-9-15的天气")
        ]
    }
)
result=response["messages"][-1].content
# print(result)
# print(response)

1 绑定2个自定义工具

In [17]:
class newsInput(BaseModel):
    """
    查询新闻工具的形参的类的相关信息
    """
    city: str=Field(description="城市名称，例如'北京市','重庆市'",default="重庆市")
    date:str=Field(description="查询日期")
news="""
问界由华为与赛力斯联合设计打造，凝聚着双方团队、产业伙伴的共同努力，也是跨界融合与产业升级的重要成果。感谢每一位消费者的选择与信赖。
问界仍然是鸿蒙智行大家庭的核心成员之一。即日起，问界将在鸿蒙智行合作框架下探索新合作模:式:产品定义、产品设计、品牌营销、渠道零售、服务体系由赛力斯主导，华为终端参与赋能。所有问界用户的既有权益及后续服务不受影响。
赛力斯与华为将继续携手并进，问界品牌专属专;营为用户带来更极致的高端智能电动汽车产品和更优:质的全生命周期用车体验
"""
@tool(description="查询具体城市的新闻",args_schema=newsInput)
def get_news(city:str,date:str):
    return f"{city} {date}的新闻有:{news}"
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    tools=[get_weather, get_news]
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                SystemMessage("你是1个私人ai助手"),
                HumanMessage("请你查询一下重庆2026-9-15的天气和新闻")
        ]
    }
)
result=response["messages"][-1].content
print(result)

以下是重庆市 **2026年9月15日** 的查询结果：

## 🌤️ 天气预报
- **天气状况**：晴朗
- **气温**：10 ~ 16℃
- **风力**：微风
- **出行建议**：天气舒适，适宜出行 👍

## 📰 当日新闻
主要聚焦于**问界品牌合作模式调整**：

1. **品牌背景**：问界由华为与赛力斯联合设计打造，凝聚了双方团队及产业伙伴的共同努力，是跨界融合与产业升级的重要成果。

2. **合作新模式**：问界仍是鸿蒙智行大家庭的核心成员。即日起在鸿蒙智行合作框架下探索新合作模式——
   - **赛力斯主导**：产品定义、产品设计、品牌营销、渠道零售、服务体系
   - **华为终端参与赋能**

3. **用户权益**：所有问界用户的既有权益及后续服务**不受影响**。

4. **未来展望**：赛力斯与华为将继续携手，问界品牌"专属专营"，为用户带来更极致的高端智能电动汽车产品和更优质的全生命周期用车体验。

---

💡 **温馨提示**：当日气温偏凉（最低10℃），建议早晚外出加件薄外套。如需查询其他城市或日期的信息，随时告诉我～
